# Customer Churn — Model Training & Evaluation
**Dataset**: Cell2Cell (Duke University / Teradata CRM) — 51,047 train rows × 58 features

This notebook:
1. Loads MLflow experiment results and builds a model leaderboard
2. Plots threshold sweep curves (Recall / Precision / F1 / Expected Savings)
3. Shows SHAP global importance for the best model
4. Renders a business impact chart (expected annual savings vs decision threshold)

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

import mlflow
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)

from configs.settings import settings
from src.churn.data.preprocess import prepare_datasets

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

REPORTS = Path('../reports')
REPORTS.mkdir(exist_ok=True)

print('Imports OK')

## 1. Load MLflow Experiment Results

In [ ]:
mlflow.set_tracking_uri(settings.mlflow.tracking_uri)

client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(settings.mlflow.experiment)

if exp is None:
    print('No MLflow experiment found. Run `make train` first, then re-run this notebook.')
else:
    runs_df = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="",
        order_by=["metrics.`val/roc_auc` DESC"],
    )
    print(f'Found {len(runs_df)} MLflow runs')
    runs_df.head()

In [ ]:
metric_cols = [
    'metrics.val/roc_auc', 'metrics.val/pr_auc',
    'metrics.val/recall',  'metrics.val/precision',
    'metrics.val/f1',      'metrics.val/expected_savings',
    'metrics.best_threshold',
]

available = [c for c in metric_cols if c in runs_df.columns]
leaderboard = (
    runs_df[['tags.mlflow.runName'] + available]
    .rename(columns=lambda c: c.replace('metrics.', '').replace('tags.mlflow.', ''))
    .dropna(subset=['val/roc_auc'])
    .sort_values('val/roc_auc', ascending=False)
    .reset_index(drop=True)
)
leaderboard.index += 1
leaderboard.columns.name = None

print('Model Leaderboard (sorted by Validation ROC-AUC)')
leaderboard.style \
    .background_gradient(subset=['val/roc_auc', 'val/pr_auc', 'val/recall'], cmap='Greens') \
    .format({'val/roc_auc': '{:.4f}', 'val/pr_auc': '{:.4f}',
             'val/recall': '{:.4f}', 'val/precision': '{:.4f}',
             'val/f1': '{:.4f}', 'val/expected_savings': '${:,.0f}',
             'best_threshold': '{:.2f}'})

## 2. Leaderboard Bar Chart

In [ ]:
if 'val/roc_auc' in leaderboard.columns:
    fig = px.bar(
        leaderboard,
        x='runName', y='val/roc_auc',
        color='val/roc_auc',
        color_continuous_scale='Greens',
        text='val/roc_auc',
        title='Model Comparison — Validation ROC-AUC',
        labels={'runName': 'Model', 'val/roc_auc': 'ROC-AUC'},
    )
    fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
    fig.update_layout(template='plotly_white', coloraxis_showscale=False, height=450)
    fig.write_html(str(REPORTS / 'model_comparison_auc.html'))
    fig.show()

## 3. Load Best Model + Validation Data

In [ ]:
models_dir   = Path(settings.model.models_dir)
meta_path    = models_dir / 'model_metadata.joblib'

if not meta_path.exists():
    print('model_metadata.joblib not found — run `make train` first')
else:
    metadata      = joblib.load(meta_path)
    model_file    = metadata.get('model_filename', 'churn_xgb_prod.joblib')
    model         = joblib.load(models_dir / model_file)
    best_threshold = float(metadata.get('best_threshold', 0.40))

    print(f"Best model  : {metadata.get('model_name')}")
    print(f"Model file  : {model_file}")
    print(f"Threshold   : {best_threshold}")
    print(f"Val ROC-AUC : {metadata.get('roc_auc')}")

In [ ]:
# Load data (uses saved preprocessor — no re-fitting)
data           = prepare_datasets()
X_val, y_val   = data['X_val'],  data['y_val']
X_test, y_test = data['X_test'], data['y_test']
feature_names  = data['feature_names']

y_prob_val  = model.predict_proba(X_val)[:, 1]
y_prob_test = model.predict_proba(X_test)[:, 1] if len(y_test) > 0 else None

print(f'Val  ROC-AUC: {roc_auc_score(y_val, y_prob_val):.4f}')
if y_prob_test is not None:
    print(f'Test ROC-AUC: {roc_auc_score(y_test, y_prob_test):.4f}')

## 4. ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
fpr, tpr, _ = roc_curve(y_val, y_prob_val)
auc_val     = roc_auc_score(y_val, y_prob_val)
axes[0].plot(fpr, tpr, lw=2, label=f'Val (AUC={auc_val:.4f})')
if y_prob_test is not None:
    fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_test)
    auc_test = roc_auc_score(y_test, y_prob_test)
    axes[0].plot(fpr_t, tpr_t, lw=2, linestyle='--', label=f'Test (AUC={auc_test:.4f})')
axes[0].plot([0,1],[0,1],'k--',lw=0.8)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend()

# PR
prec, rec, _ = precision_recall_curve(y_val, y_prob_val)
ap_val = average_precision_score(y_val, y_prob_val)
axes[1].plot(rec, prec, lw=2, label=f'Val (AP={ap_val:.4f})')
if y_prob_test is not None:
    prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_test)
    ap_test = average_precision_score(y_test, y_prob_test)
    axes[1].plot(rec_t, prec_t, lw=2, linestyle='--', label=f'Test (AP={ap_test:.4f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.suptitle(f'Best Model: {metadata.get("model_name")}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'roc_pr_curves.png', dpi=150)
plt.show()

## 5. Threshold Sweep — Recall / Precision / F1 vs Threshold

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.linspace(0.10, 0.70, 60)
recalls, precisions, f1s, n_flagged = [], [], [], []

for t in thresholds:
    y_pred = (y_prob_val >= t).astype(int)
    recalls.append(recall_score(y_val, y_pred, zero_division=0))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))
    f1s.append(f1_score(y_val, y_pred, zero_division=0))
    n_flagged.append(y_pred.sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds, recalls,    label='Recall',    lw=2)
axes[0].plot(thresholds, precisions, label='Precision', lw=2)
axes[0].plot(thresholds, f1s,        label='F1',        lw=2, linestyle='--')
axes[0].axvline(best_threshold, color='red', lw=1.2, linestyle=':', label=f'Threshold={best_threshold}')
axes[0].set_xlabel('Decision Threshold'); axes[0].set_ylabel('Score')
axes[0].set_title('Recall / Precision / F1 vs Threshold')
axes[0].legend()

axes[1].plot(thresholds, n_flagged, color='darkorange', lw=2)
axes[1].axvline(best_threshold, color='red', lw=1.2, linestyle=':')
axes[1].set_xlabel('Decision Threshold'); axes[1].set_ylabel('Customers Flagged')
axes[1].set_title('Customers Flagged for Retention at Each Threshold')

plt.suptitle('Threshold Sweep Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'threshold_sweep.png', dpi=150)
plt.show()

## 6. Business Impact — Expected Annual Savings vs Threshold

In [ ]:
AVG_ANNUAL_VALUE   = 48.46 * 12
CALL_COST          = 10.0
SUCCESS_RATE       = 0.30

savings = []
for t in thresholds:
    y_pred = (y_prob_val >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    s = tp * AVG_ANNUAL_VALUE * SUCCESS_RATE - (tp + fp) * CALL_COST
    savings.append(s)

best_idx = int(np.argmax(savings))
best_t_econ = thresholds[best_idx]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=thresholds, y=savings,
    mode='lines', line=dict(width=3),
    name='Expected Savings',
))
fig.add_vline(x=best_t_econ, line=dict(color='red', dash='dot'),
              annotation_text=f'Optimal: {best_t_econ:.2f}', annotation_position='top right')
fig.add_vline(x=best_threshold, line=dict(color='green', dash='dot'),
              annotation_text=f'Model default: {best_threshold}', annotation_position='top left')
fig.update_layout(
    title='Expected Annual Savings vs Decision Threshold (Validation Set)',
    xaxis_title='Threshold', yaxis_title='Expected Savings ($)',
    template='plotly_white', height=450,
)
fig.write_html(str(REPORTS / 'business_impact.html'))
fig.show()

print(f'Optimal economic threshold : {best_t_econ:.2f}')
print(f'Expected savings @ optimal : ${max(savings):,.0f}')
print(f'Expected savings @ {best_threshold}    : ${savings[np.argmin(np.abs(thresholds - best_threshold))]:,.0f}')

## 7. Confusion Matrix at Best Threshold

In [ ]:
y_pred_val = (y_prob_val >= best_threshold).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (split, y_true, y_pred_) in zip(
    axes,
    [
        ('Validation', y_val,  y_pred_val),
        ('Test',        y_test, (y_prob_test >= best_threshold).astype(int))
        if y_prob_test is not None else ('Test', y_val, y_pred_val),
    ],
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred_),
        display_labels=['No Churn', 'Churn'],
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {split} (t={best_threshold})')

plt.tight_layout()
plt.savefig(REPORTS / 'confusion_matrix.png', dpi=150)
plt.show()

## 8. SHAP Feature Importance

In [ ]:
import shap

model_name_safe = metadata.get('model_name', '').lower().replace(' ', '_')
shap_path = models_dir / f'shap_explainer_{model_name_safe}.joblib'

if shap_path.exists():
    explainer   = joblib.load(shap_path)
    sample_size = min(2000, X_val.shape[0])
    rng         = np.random.default_rng(42)
    idx         = rng.choice(X_val.shape[0], sample_size, replace=False)
    X_sample    = X_val[idx]

    shap_values = explainer.shap_values(X_sample)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    # Beeswarm
    fig, _ = plt.subplots(figsize=(9, 7))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      max_display=20, show=False)
    plt.title('SHAP Beeswarm — Top 20 Feature Impacts', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORTS / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Bar chart
    fig, _ = plt.subplots(figsize=(9, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      max_display=20, plot_type='bar', show=False)
    plt.title('SHAP Feature Importance (mean |SHAP|)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORTS / 'shap_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'SHAP explainer not found at {shap_path}')
    print('Re-run `make train` or run the train script to generate SHAP explainers.')

## 9. Top Feature Importance Table

In [ ]:
if shap_path.exists():
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance_df = (
        pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_abs_shap})
        .sort_values('mean_abs_shap', ascending=False)
        .reset_index(drop=True)
    )
    importance_df.index += 1

    print(f'Top 20 features by mean |SHAP| ({metadata.get("model_name")})')
    importance_df.head(20).style \
        .background_gradient(subset=['mean_abs_shap'], cmap='YlOrRd') \
        .format({'mean_abs_shap': '{:.5f}'})

## 10. Load Threshold Sweep CSVs from MLflow

In [ ]:
# Load threshold sweep CSVs logged to MLflow during training
sweep_files = sorted(Path('..').rglob('threshold_sweep_*.csv'))

if sweep_files:
    sweeps = {}
    for fp in sweep_files:
        name = fp.stem.replace('threshold_sweep_', '')
        sweeps[name] = pd.read_csv(fp)

    fig = go.Figure()
    for name, df in sweeps.items():
        fig.add_trace(go.Scatter(
            x=df['threshold'], y=df['recall'],
            mode='lines+markers', name=name,
        ))
    fig.update_layout(
        title='Churn Recall vs Threshold — All Models',
        xaxis_title='Threshold', yaxis_title='Recall',
        template='plotly_white', height=450,
    )
    fig.show()
else:
    print('No threshold_sweep_*.csv files found — run `make train` first')

## Summary

| Step | Result |
|------|--------|
| Best model | See leaderboard above |
| Decision threshold | Set by maximising expected annual savings |
| SHAP top driver | `MonthsInService`, `DroppedCalls`, `MadeCallToRetentionTeam` (typical) |
| Deployment | `make docker-up` → FastAPI on port 8000 |
| AWS | `terraform apply` in `infrastructure/`, then `git push main` triggers CI/CD |